# Telco Customer Churn: Black Box Robustness Audit

The two models trained in `train_churn.ipynb` are loaded here as black boxes. The auditor only calls `predict_proba`, nothing else. The test set gets corrupted with increasing amounts of noise and we track how much worse each model gets. The end result is a degradation curve per model.

## Setup

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, average_precision_score
import os


## Rebuild the test set

Same split as the training notebook (same test_size, stratify and random_state), so we get the identical held out test set the models were evaluated on. The encoding is locked to the training columns with reindex, so corrupted copies always come out with the same 30 columns the models expect.

In [ ]:
df_churn = pd.read_csv("../data/processed/churn_clean.csv")

In [ ]:
df_clean = df_churn.drop("customerID", axis=1)

x = df_clean.drop("Churn", axis=1)
y = df_clean["Churn"].map({"Yes": 1, "No": 0})

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    x, y, test_size=0.25, stratify=y, random_state=42
)

In [ ]:
X_test_raw = X_test.copy()

X_train = pd.get_dummies(X_train, drop_first=True, dtype=int)
TRAIN_COLUMNS = X_train.columns


def encode_like(df_raw):
    return pd.get_dummies(df_raw, drop_first=True, dtype=int).reindex(
        columns=TRAIN_COLUMNS, fill_value=0
    )


X_test = encode_like(X_test_raw)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Feature groups

Three numeric columns, the rest categorical. SeniorCitizen is stored as 0/1 but it is really a flag, so it goes with the categoricals. Swapping a flag makes sense, multiplying it by noise does not.

In [ ]:
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]

## Load the frozen models

Loaded from outputs/models. No coefficients or feature importances are inspected anywhere in this notebook.

In [ ]:
models = {
    "Logistic Regression": joblib.load("../outputs/models/churn_logistic_regression.joblib"),
    "XGBoost": joblib.load("../outputs/models/churn_xgboost.joblib"),
}

## Scoring helper

F1 needs hard labels so the probabilities are thresholded at 0.5. ROC-AUC and PR-AUC work on the probabilities directly.

In [ ]:
def audit_metrics(model, X, y_true):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    return {
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }

## Clean baselines

With no noise both models should reproduce the training notebook scores, around 0.84 ROC-AUC. If they don't, the split replay is broken and nothing after this cell can be trusted.

In [ ]:
for model_name, model in models.items():
    print(f"{model_name}: {audit_metrics(model, X_test, Y_test)}")

## Perturbation engine

Numeric columns get multiplicative noise: each value is multiplied by 1 + U(-rate, rate).

Categorical columns get swaps: exactly rate percent of the rows are picked at random and their value is replaced with a draw from the column's own distribution. The draw never returns the value the row already had, so a swap always changes something.

One shared RNG drives all of it. Each noise level is corrupted once and both models score the same corrupted copy, which keeps the comparison fair.

In [ ]:
def perturb_numeric(df, features, rate, rng):
    df_perturbed = df.copy()
    for feature in features:
        noise = 1 + rng.uniform(-rate, rate, size=len(df_perturbed))
        df_perturbed[feature] = df_perturbed[feature] * noise
    return df_perturbed

In [ ]:
def perturb_categorical(df, features, rate, rng):
    df_perturbed = df.copy()
    for feature in features:
        probs = df_perturbed[feature].value_counts(normalize=True)
        n_swap = int(round(len(df_perturbed) * rate))
        swap_idx = rng.choice(len(df_perturbed), size=n_swap, replace=False)
        values = df_perturbed[feature].to_numpy(copy=True)
        for i in swap_idx:
            p = probs.copy()
            p[values[i]] = 0.0
            p = p / p.sum()
            values[i] = rng.choice(p.index, p=p.values)
        df_perturbed[feature] = values
    return df_perturbed

In [ ]:
def corrupt_data(df, numeric_features, categorical_features, numeric_rate, categorical_rate, rng):
    df_perturbed = perturb_numeric(df, numeric_features, numeric_rate, rng)
    df_perturbed = perturb_categorical(df_perturbed, categorical_features, categorical_rate, rng)
    return df_perturbed

## Audit loop

For every noise level: corrupt the raw test frame, encode it, score both models, record the numbers. Level 0 is the clean reference. The RNG is seeded, so reruns give the same numbers.

In [ ]:
NOISE_LEVELS = [0.00, 0.05, 0.10, 0.20]

rng = np.random.default_rng(seed=42)

records = []
for level in NOISE_LEVELS:
    if level == 0:
        dirty_frame = X_test_raw
    else:
        dirty_frame = corrupt_data(
            X_test_raw, numerical_features, categorical_features, level, level, rng
        )
    encoded = encode_like(dirty_frame)
    for name, model in models.items():
        metrics = audit_metrics(model, encoded, Y_test)
        metrics["model"] = name
        metrics["noise_level"] = level
        records.append(metrics)

results = pd.DataFrame(records)
results

## Relative degradation

Each score divided by that model's own clean score, expressed as a percentage drop. This makes the two models comparable even though their clean baselines differ.

In [ ]:
clean = results[results["noise_level"] == 0.0][["model", "roc_auc", "f1", "pr_auc"]].rename(
    columns={"roc_auc": "roc_auc_clean", "f1": "f1_clean", "pr_auc": "pr_auc_clean"}
)

merged = results.merge(clean, on="model")
for metric in ["roc_auc", "f1", "pr_auc"]:
    merged[f"{metric}_pct_drop"] = (1 - merged[metric] / merged[f"{metric}_clean"]) * 100

table = merged[merged["noise_level"] > 0].pivot(
    index="noise_level", columns="model",
    values=["roc_auc_pct_drop", "f1_pct_drop", "pr_auc_pct_drop"],
)
table.round(2)

## Save results

One row per model per noise level, written to outputs/results. Gitignored like all artifacts, the audit reruns from the notebook and the seed.

In [ ]:
os.makedirs("../outputs/results", exist_ok=True)
results.to_csv("../outputs/results/churn_audit_results.csv", index=False)
print("Results saved.")

## Degradation curves

The dashed line is each model's clean baseline. The label at the end of each curve is the total drop at 20% noise.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {"Logistic Regression": "#1f77b4", "XGBoost": "#d62728"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=120)

for ax, metric, title in zip(
    axes,
    ["roc_auc", "f1", "pr_auc"],
    ["ROC-AUC", "F1 (threshold 0.5)", "PR-AUC"],
):
    for name in models:
        sub = results[results["model"] == name].sort_values("noise_level")
        baseline = sub.iloc[0][metric]
        ax.axhline(baseline, color=COLORS[name], linestyle="--", linewidth=1, alpha=0.5)
        ax.plot(
            sub["noise_level"], sub[metric],
            marker="o", markersize=6, linewidth=2.5,
            color=COLORS[name], label=name,
        )
        last = sub.iloc[-1]
        drop = (1 - last[metric] / baseline) * 100
        ax.annotate(
            f"{drop:.1f}%",
            (last["noise_level"], last[metric]),
            textcoords="offset points", xytext=(8, -4),
            fontsize=10, color=COLORS[name], fontweight="bold",
        )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Noise level", fontsize=12)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_ylim(0.35, 0.95)
    ax.set_xticks([0.0, 0.05, 0.10, 0.20])
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=10)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02),
           ncol=2, frameon=False, fontsize=12)

fig.suptitle("Churn models: performance under feature noise", fontsize=15, fontweight="bold", y=1.12)
fig.tight_layout()
os.makedirs("../outputs/figures", exist_ok=True)
fig.savefig("../outputs/figures/churn_degradation_curves.png", dpi=150, bbox_inches="tight")
plt.show()